# DSA Week 3 -- Searching: Linear vs Binary

**Course:** Data Structures & Algorithms
**Session:** 3 hours
**Prerequisites:** Weeks 1-2 (Big-O, Benchmarking)
**Focus:** Linear search, binary search, sorted data

## Learning Objectives

1. Implement linear search and understand its O(n) cost
2. Implement binary search and understand its O(log n) cost
3. Trace binary search step-by-step on paper
4. Know the precondition: data must be sorted for binary search
5. Use Python's `bisect` module for production binary search
6. Benchmark linear vs binary and see the difference at scale

## The Big Idea

Searching is the most common operation in computing. Every time your pipeline
looks up a record, finds a threshold crossing, or checks if a value exists,
it is searching. The difference between O(n) and O(log n) can be enormous:

```
n = 1,000,000 items:
  Linear search: up to 1,000,000 comparisons
  Binary search: at most 20 comparisons

That is 50,000x fewer operations!
```

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Linear Search -- The Slow but Simple Way

Linear search checks every element one by one until it finds the target
(or reaches the end). It works on **any** data -- sorted or unsorted.

```
Linear search for target = 42:

  [15, 23, 8, 42, 16, 50, 4, 31]
   ^   no
       ^   no
           ^  no
              ^   FOUND at index 3!

  Worst case: check ALL elements -> O(n)
  Best case:  first element matches -> O(1)
  Average:    check n/2 elements -> O(n)
```

In [ ]:
def linear_search(data, target):
    """Search by checking every element. O(n)."""
    for i, item in enumerate(data):
        if item == target:
            return i
    return -1

# === Example 1: Basic search ===
data = [15, 23, 8, 42, 16, 50, 4, 31]
print("Data:", data)
print()

for target in [42, 50, 99]:
    idx = linear_search(data, target)
    if idx >= 0:
        print("  Search for " + str(target) + ": FOUND at index " + str(idx))
    else:
        print("  Search for " + str(target) + ": NOT FOUND")

**Expected Output:**
```
Data: [15, 23, 8, 42, 16, 50, 4, 31]

  Search for 42: FOUND at index 3
  Search for 50: FOUND at index 5
  Search for 99: NOT FOUND
```

### Example 2: Traced Linear Search

Let us see exactly what happens at each step.

In [ ]:
def linear_search_traced(data, target):
    """Linear search with step-by-step trace."""
    print("  Searching for " + str(target) + " in " + str(data))
    for i, item in enumerate(data):
        status = "MATCH!" if item == target else "skip"
        print("    Step " + str(i + 1) + ": compare data[" + str(i) + "]=" + str(item) + " with " + str(target) + " -> " + status)
        if item == target:
            print("  Found at index " + str(i) + " in " + str(i + 1) + " steps")
            return i
    print("  Not found after " + str(len(data)) + " steps")
    return -1

data = [15, 23, 8, 42, 16, 50, 4, 31]
print("=== Trace 1: target exists ===")
linear_search_traced(data, 42)
print()
print("=== Trace 2: target not found ===")
linear_search_traced(data, 99)

**Expected Output:**
```
=== Trace 1: target exists ===
  Searching for 42 in [15, 23, 8, 42, 16, 50, 4, 31]
    Step 1: compare data[0]=15 with 42 -> skip
    Step 2: compare data[1]=23 with 42 -> skip
    Step 3: compare data[2]=8 with 42 -> skip
    Step 4: compare data[3]=42 with 42 -> MATCH!
  Found at index 3 in 4 steps

=== Trace 2: target not found ===
  Searching for 99 in [15, 23, 8, 42, 16, 50, 4, 31]
    Step 1: compare data[0]=15 with 99 -> skip
    Step 2: compare data[1]=23 with 99 -> skip
    ...
    Step 8: compare data[7]=31 with 99 -> skip
  Not found after 8 steps
```

---
## Part 2: Binary Search -- The Fast Way (Requires Sorted Data!)

Binary search is like the number-guessing game:
"I am thinking of a number 1-100. You guess. I say higher or lower."

The optimal strategy: always guess the MIDDLE. This halves the remaining
possibilities each time.

```
Binary search for target = 9 in [1, 3, 5, 7, 9, 11, 13, 15]:

Step 1: [1, 3, 5, |7|, 9, 11, 13, 15]    mid=7    9 > 7 -> go RIGHT
                         ^^^^^^^^^^^^^
Step 2:              [9, |11|, 13, 15]     mid=11   9 < 11 -> go LEFT
                      ^
Step 3:              [|9|]                  mid=9    FOUND!

Only 3 steps for 8 items! (log2(8) = 3)

Compare: linear search would take up to 8 steps.
For 1,000,000 items: binary = 20 steps, linear = 1,000,000 steps!
```

**CRITICAL PRECONDITION:** Binary search ONLY works on sorted data.
If the data is not sorted, the results will be wrong.

In [ ]:
def binary_search(sorted_data, target):
    """Binary search on sorted data. O(log n).

    PRECONDITION: sorted_data must be sorted in ascending order!
    """
    low = 0
    high = len(sorted_data) - 1
    steps = 0

    while low <= high:
        steps += 1
        mid = (low + high) // 2

        if sorted_data[mid] == target:
            return mid, steps
        elif sorted_data[mid] < target:
            low = mid + 1    # target is in the RIGHT half
        else:
            high = mid - 1   # target is in the LEFT half

    return -1, steps  # not found

# === Example 1: Basic binary search ===
data = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
print("Sorted data:", data)
print()

for target in [9, 1, 19, 8]:
    idx, steps = binary_search(data, target)
    if idx >= 0:
        print("  Search for " + str(target).rjust(2) + ": FOUND at index " + str(idx) + " in " + str(steps) + " steps")
    else:
        print("  Search for " + str(target).rjust(2) + ": NOT FOUND in " + str(steps) + " steps")

**Expected Output:**
```
Sorted data: [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]

  Search for  9: FOUND at index 4 in 3 steps
  Search for  1: FOUND at index 0 in 3 steps
  Search for 19: FOUND at index 9 in 4 steps
  Search for  8: NOT FOUND in 4 steps
```

### Example 2: Step-by-Step Binary Search Trace

In [ ]:
def binary_search_traced(sorted_data, target):
    """Binary search with ASCII art trace."""
    low = 0
    high = len(sorted_data) - 1
    step = 0

    print("  Target: " + str(target))
    print("  Data:   " + str(sorted_data))
    print()

    while low <= high:
        step += 1
        mid = (low + high) // 2

        # Build visual
        markers = [" "] * len(sorted_data)
        for i in range(low, high + 1):
            markers[i] = "-"
        markers[mid] = "^"

        # Show active range
        active = str(sorted_data[low:high + 1])
        comparison = ""
        if sorted_data[mid] == target:
            comparison = str(sorted_data[mid]) + " == " + str(target) + " FOUND!"
        elif sorted_data[mid] < target:
            comparison = str(sorted_data[mid]) + " < " + str(target) + " -> go RIGHT"
        else:
            comparison = str(sorted_data[mid]) + " > " + str(target) + " -> go LEFT"

        print("  Step " + str(step) + ": low=" + str(low) + " high=" + str(high) + " mid=" + str(mid))
        print("         mid value = " + str(sorted_data[mid]) + " | " + comparison)
        print()

        if sorted_data[mid] == target:
            return mid, step
        elif sorted_data[mid] < target:
            low = mid + 1
        else:
            high = mid - 1

    print("  Not found after " + str(step) + " steps")
    return -1, step

data = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("=== Binary Search Trace ===")
print()
binary_search_traced(data, 23)

**Expected Output:**
```
=== Binary Search Trace ===

  Target: 23
  Data:   [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]

  Step 1: low=0 high=9 mid=4
         mid value = 16 | 16 < 23 -> go RIGHT

  Step 2: low=5 high=9 mid=7
         mid value = 56 | 56 > 23 -> go LEFT

  Step 3: low=5 high=6 mid=5
         mid value = 23 | 23 == 23 FOUND!
```

---
## Part 3: Linear vs Binary -- Head-to-Head at Scale

Now let us see the real difference with large data.

In [ ]:
import math

print("=== Operation Count Comparison ===")
print()
print("  " + "n".rjust(12) + " | " + "Linear (worst)".rjust(15) + " | " + "Binary (worst)".rjust(15) + " | " + "Ratio".rjust(10))
print("  " + "-" * 12 + "-|-" + "-" * 15 + "-|-" + "-" * 15 + "-|-" + "-" * 10)

for n in [10, 100, 1_000, 10_000, 100_000, 1_000_000, 100_000_000]:
    linear = n
    binary = math.ceil(math.log2(n)) if n > 0 else 0
    ratio = linear / binary if binary > 0 else 0
    print("  " + "{:>12,}".format(n) + " | " + "{:>15,}".format(linear) + " | " + "{:>15,}".format(binary) + " | " + "{:>8,.0f}x".format(ratio))

**Expected Output:**
```
=== Operation Count Comparison ===

             n |  Linear (worst) |  Binary (worst) |      Ratio
  -------------|-----------------|-----------------|----------
            10 |              10 |               4 |        2x
           100 |             100 |               7 |       14x
         1,000 |           1,000 |              10 |      100x
        10,000 |          10,000 |              14 |      714x
       100,000 |         100,000 |              17 |    5,882x
     1,000,000 |       1,000,000 |              20 |   50,000x
   100,000,000 |     100,000,000 |              27 |3,703,704x
```

At 100 million items, binary search needs only **27 comparisons**.
Linear search needs up to **100 million**. That is a 3.7 million times difference.

---
## Part 4: Timing Experiment

In [ ]:
import timeit

def time_searches(n, n_runs=1000):
    """Compare linear and binary search timing at size n."""
    data = list(range(n))
    target = n - 1  # worst case for linear

    t_lin = timeit.timeit(lambda: linear_search(data, target), number=n_runs)
    t_bin = timeit.timeit(lambda: binary_search(data, target), number=n_runs)

    lin_us = t_lin / n_runs * 1e6
    bin_us = t_bin / n_runs * 1e6
    return lin_us, bin_us

print("=== Timing: Linear vs Binary Search ===")
print()
print("  " + "n".rjust(10) + " | " + "Linear (us)".rjust(12) + " | " + "Binary (us)".rjust(12) + " | " + "Speedup".rjust(10))
print("  " + "-" * 10 + "-|-" + "-" * 12 + "-|-" + "-" * 12 + "-|-" + "-" * 10)

for n in [100, 1_000, 10_000, 100_000]:
    runs = max(100, 10_000 // n * 100)
    lin_us, bin_us = time_searches(n, n_runs=runs)
    speedup = lin_us / bin_us if bin_us > 0 else 0
    print("  " + "{:>10,}".format(n) + " | " + "{:>10.1f}us".format(lin_us) + " | " + "{:>10.1f}us".format(bin_us) + " | " + "{:>8.0f}x".format(speedup))

---
## Part 5: Python's `bisect` Module -- Production Binary Search

In production code, use Python's built-in `bisect` module instead of writing
your own binary search. It is implemented in C and highly optimized.

In [ ]:
import bisect

sorted_data = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]

# bisect_left: find insertion point (leftmost position)
idx = bisect.bisect_left(sorted_data, 23)
print("bisect_left(23) = " + str(idx) + " -> value = " + str(sorted_data[idx]))

# Use for exact search:
def bisect_search(sorted_data, target):
    """Binary search using bisect. O(log n)."""
    idx = bisect.bisect_left(sorted_data, target)
    if idx < len(sorted_data) and sorted_data[idx] == target:
        return idx
    return -1

print()
for target in [23, 8, 99]:
    idx = bisect_search(sorted_data, target)
    if idx >= 0:
        print("  Search " + str(target) + ": found at index " + str(idx))
    else:
        print("  Search " + str(target) + ": not found")

# Range query: find all values in [10, 50]
left = bisect.bisect_left(sorted_data, 10)
right = bisect.bisect_right(sorted_data, 50)
print()
print("Values in range [10, 50]: " + str(sorted_data[left:right]))

**Expected Output:**
```
bisect_left(23) = 5 -> value = 23

  Search 23: found at index 5
  Search 8: found at index 2
  Search 99: not found

Values in range [10, 50]: [12, 16, 23, 38]
```

### When to Use This

| Situation | Use |
|-----------|-----|
| Data is unsorted, search once | Linear search O(n) |
| Data is unsorted, search many times | Sort first O(n log n) + binary search O(log n) each |
| Data is sorted | Always binary search O(log n) |
| Need range queries on sorted data | `bisect_left` + `bisect_right` |
| Need to check membership in any order | Use a set O(1) |

---
## Common Mistakes

| Mistake | Why | Fix |
|---------|-----|-----|
| Binary search on unsorted data | Results are wrong! | Sort first, or use linear search |
| Off-by-one in low/high | Infinite loop or missed element | Use `low <= high` (not `<`) |
| Integer overflow in mid | `(low + high)` can overflow in other languages | In Python, integers are arbitrary precision -- not an issue |
| Forgetting to return -1 | Function returns None for not-found | Always have an explicit not-found return |

---
## Mini-Quiz

In [ ]:
# Q1: You have 1 billion sorted numbers. How many comparisons
# does binary search need at most?
import math
# Answer: math.ceil(math.log2(1_000_000_000)) = ___

# Q2: Your data is NOT sorted. Is it worth sorting it just
# to do one binary search? Why or why not?
# Answer:

# Q3: You need to search for many values in a large dataset.
# What data structure should you use instead of binary search?
# Answer:

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)